In [2]:
import os
import gc
import zarr
import yaml
import json
import numba
import numpy as np
import polars as pl
import pandas as pd
from tqdm import tqdm
import statsmodels.api as sm

from plotnine import *
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from scipy.stats import spearmanr

In [3]:
RAP_ANNO_DIR = "project-Gyp4fvjJg0yFZ374KvP9bGFJ:/processed_data/wgs/qced_maf1e-3_loftee_olink_genes/qced_maf1e-3_loftee_olink_genes_EURunrelated"
LOCAL_ANNO_DIR = "/home/dnanexus/data_dir"

# ANNO_FILE = "annotations_fillna_ukbgym.parquet"
ANNO_FILE = "annotations_fillna_ukbgym_with_mane.parquet"

!dx download {RAP_ANNO_DIR}/{ANNO_FILE} -o {LOCAL_ANNO_DIR}/{ANNO_FILE}
anno = pl.read_parquet(f"{LOCAL_ANNO_DIR}/{ANNO_FILE}")
anno

Error: path
"/home/dnanexus/data_dir/annotations_fillna_ukbgym_with_mane.parquet" already
exists but -f/--overwrite was not set


id,chrom,pos,ref,alt,region,sift,polyphen,cadd_phred,cadd_raw,am_pathogenicity,spliceai_pred,loftee_hc,loftee_lc,relative_cds_position,next_in_frame_relative,spliceai_delta_score,pangolin_score,delta_score,absplice_dna_max,absplice2_max,five_prime_utr_variant_consequence_uaug_gained,five_prime_utr_variant_consequence_uaug_lost,five_prime_utr_variant_consequence_uframeshift,five_prime_utr_variant_consequence_ustop_gained,five_prime_utr_variant_consequence_ustop_lost,variant_length,gpn_score,promoterai,score_pai3d,abexp_abs_max,gene_length,dist_to_tss,aparent2,cdspos,cpg,dst2splice,…,relprotpos_is_na,relcdnapos_is_na,toverlapmotifs_is_na,targetscan_is_na,verphcons_is_na,verphylop_is_na,sc_ukb,ac_ukb,mac_ukb,vep_cds_relaxed,tss,strand,gene_length_right,gene_name,dist_to_tss_right,region_right,protein_position,amino_acids,mobi_curated_disorder_priority,mobi_lip_full,ted_domain,low_complexity_domain,not_annotated_in_encode,encode_dels,encode_ca-ctcf,encode_ca,encode_ca-h3k4me3,encode_tf,encode_ca-tf,encode_pels,encode_pls,mane_exonic,mane_cds,mane_utr,non_mane_exonic,non_mane_cds,non_mane_utr
str,str,i64,str,str,str,f32,f32,f32,f32,f32,str,i8,i8,f32,f32,f32,f32,f32,f32,f32,u8,u8,u8,u8,u8,u32,f32,f32,f32,f32,i64,i64,f32,f32,f32,f32,…,i8,i8,i8,i8,i8,i8,u64,i64,i64,bool,i64,cat,i64,str,i64,str,str,str,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool
"""chr8:134702216:A:G""","""chr8""",134702216,"""A""","""G""","""ENSG00000066827""",1.0,0.0,2.22,0.191705,0.0,"""ZFAT|0.00|0.00|0.00|0.00|11|-2…",0,0,0.0,0.0,0.0,0.0,0.0,0.001,0.000033,0,0,0,0,0,1,-0.27,0.0,0.0,0.005598,235263,10833,0.0,0.0,0.0,0.0,…,1,1,1,1,0,0,2,2,2,false,134713049,"""-""",235263,"""ZFAT""",10833,"""ENSG00000066827""",null,null,false,false,false,false,true,false,false,false,false,false,false,false,false,false,false,false,false,false,false
"""chr19:47846298:C:T""","""chr19""",47846298,"""C""","""T""","""ENSG00000105392""",1.0,0.0,3.403,0.31197,0.0,"""0""",0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0,0,0,0,1,-0.42,0.0,0.0,0.0,23553,26520,0.0,0.0,0.067,0.0,…,1,1,1,1,0,0,1,1,1,false,47819778,"""+""",23553,"""CRX""",26520,"""ENSG00000105392""",null,null,false,false,false,false,true,false,false,false,false,false,false,false,false,false,false,false,false,false,false
"""chr17:49763088:G:A""","""chr17""",49763088,"""G""","""A""","""ENSG00000121104""",1.0,0.0,10.93,1.054279,0.0,"""FAM117A|0.00|0.00|0.00|0.00|-3…",0,0,0.0,0.0,0.0,0.0,0.0,0.001,0.000033,0,0,0,0,0,1,-1.25,0.0,0.0,0.013111,78850,26092,0.0,0.0,0.04,0.0,…,1,1,1,1,0,0,2,2,2,false,49789180,"""-""",78850,"""FAM117A""",26092,"""ENSG00000121104""",null,null,false,false,false,false,false,false,false,false,false,false,false,true,false,false,false,false,false,false,false
"""chr8:38172782:G:C""","""chr8""",38172782,"""G""","""C""","""ENSG00000175324""",1.0,0.0,0.644,-0.129746,0.0,"""LSM1|0.00|0.00|0.00|0.00|0|-22…",0,0,0.0,0.0,0.0,0.0,0.0,0.001,0.000033,0,0,0,0,0,1,-1.67,0.0,0.0,0.005501,13397,3948,0.0,0.0,0.0,0.0,…,1,1,0,1,0,0,1,1,1,false,38176730,"""-""",13397,"""LSM1""",3948,"""ENSG00000156735""",null,null,false,false,false,false,true,false,false,false,false,false,false,false,false,false,false,false,false,false,false
"""chr1:174771515:C:T""","""chr1""",174771515,"""C""","""T""","""ENSG00000152061""",1.0,0.0,6.383,0.592475,0.0,"""RABGAP1L|0.00|0.00|0.00|0.00|3…",0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0,0,0,0,1,-1.52,0.0,0.0,0.006189,835900,612106,0.0,0.0,0.0,0.0,…,1,1,1,1,0,0,1,1,1,false,174159409,"""+""",835900,"""RABGAP1L""",612106,"""ENSG00000152061""",null,null,false,false,false,false,true,false,false,false,false,false,false,false,false,false,false,false,false,false,false
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""chr11:70472380:A:T""","""chr11""",70472380,"""A""","""T""","""ENSG00000162105""",1.0,0.0,0.45,-0.21966,0.0,"""SHANK2|0.00|0.00|0.00|0.00|-2|…",0,0,0.0,0.0,0.0,0.0,0.0,0.0,0

In [6]:
fz_file_prefix = "TSS_flashzoi_EURunrelated_variants_burden_test_olink_genes_GENCODEv40"
!dx download project-Gyp4fvjJg0yFZ374KvP9bGFJ:/processed_data/ukbgym/flashzoi_scores/{fz_file_prefix}* -o /home/dnanexus/data_dir/

Error: path "/home/dnanexus/data_dir/TSS_flashzoi_EURunrelated_variants_burden
_test_olink_genes_GENCODEv403.pt" already exists but -f/--overwrite was not
set


In [7]:
a = pl.read_parquet(f'/home/dnanexus/data_dir/{fz_file_prefix}_3.parquet')
a

Chromosome,Start,End,gene_name,Strand,Pos,Ref,Alt,gene_id,transcript_id,burden_gene,olink_gene,promoterAI,MANE_status,tss_pos,dist_to_tss,variant,unique_id
str,i64,i64,str,str,i64,str,str,str,str,bool,bool,f64,str,i64,i64,str,str
"""chr1""",757975,1282263,"""AGRN""","""+""",1019650,"""G""","""A""","""ENSG00000188157""","""ENST00000379370.7""",false,true,0.0328,"""MANE Select""",1020119,468,"""chr1_1019650_G_A""","""chr1_1019650_G_A_ENSG000001881…"
"""chr1""",757975,1282263,"""AGRN""","""+""",1019763,"""A""","""T""","""ENSG00000188157""","""ENST00000379370.7""",false,true,0.0908,"""MANE Select""",1020119,355,"""chr1_1019763_A_T""","""chr1_1019763_A_T_ENSG000001881…"
"""chr1""",757975,1282263,"""AGRN""","""+""",1019639,"""G""","""A""","""ENSG00000188157""","""ENST00000379370.7""",false,true,0.0388,"""MANE Select""",1020119,479,"""chr1_1019639_G_A""","""chr1_1019639_G_A_ENSG000001881…"
"""chr1""",757975,1282263,"""AGRN""","""+""",1019620,"""G""","""A""","""ENSG00000188157""","""ENST00000379370.7""",false,true,-0.0876,"""MANE Select""",1020119,498,"""chr1_1019620_G_A""","""chr1_1019620_G_A_ENSG000001881…"
"""chr1""",757975,1282263,"""AGRN""","""+""",1019690,"""C""","""G""","""ENSG00000188157""","""ENST00000379370.7""",false,true,-0.0236,"""MANE Select""",1020119,428,"""chr1_1019690_C_G""","""chr1_1019690_C_G_ENSG000001881…"
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""chr22""",50366029,50890317,"""ARSA""","""-""",50628450,"""G""","""C""","""ENSG00000100299""","""ENST00000216124.10""",false,true,-0.0079,"""MANE Select""",50628173,-278,"""chr22_50628450_G_C""","""chr22_50628450_G_C_ENSG0000010…"
"""chr22""",50366029,50890317,"""ARSA""","""-""",50627787,"""G""","""C""","""ENSG00000100299""","""ENST00000216124.10""",false,true,0.1016,"""MANE Select""",50628173,385,"""chr22_50627787_G_C""","""chr22_50627787_G_C_ENSG0000010…"
"""chr22""",50366029,50890317,"""ARSA""","""-""",50628287,"""T""","""C""","""ENSG00000100299""","""ENST00000216124.10""",false,true,-0.0326,"""MANE Select""",50628173,-115,"""chr22_50628287_T_C""","""chr22_50628287_T_C_ENSG0000010…"


In [8]:
!dx download project-Gyp4fvjJg0yFZ374KvP9bGFJ:/processed_data/ukbgym/flashzoi_scores/targets.txt -o /home/dnanexus/data_dir/

tracks_df = pd.read_table('/home/dnanexus/data_dir/targets.txt')
descriptions = tracks_df.loc[(tracks_df.description.str.contains("RNA:")) & (~tracks_df.identifier.str.endswith("-")) ][['description', 'strand_pair']]
descriptions['description_unique'] = descriptions['description'] + '_' + descriptions['strand_pair'].astype(str)
descriptions

[===========================================================>] Completed 1,215,560 of 1,215,560 bytes (100%) /home/dnanexus/data_dir/targets.txtt


,description,strand_pair,description_unique
6068,RNA:aortic smooth muscle cell male adult (21 y...,6069,RNA:aortic smooth muscle cell male adult (21 y...
6070,RNA:bladder microvascular endothelial cell mal...,6071,RNA:bladder microvascular endothelial cell mal...
6072,RNA:smooth muscle cell of bladder female adult...,6073,RNA:smooth muscle cell of bladder female adult...
6074,RNA:bronchial epithelial cell female adult (40...,6075,RNA:bronchial epithelial cell female adult (40...
6076,RNA:bronchial smooth muscle cell male adult (5...,6077,RNA:bronchial smooth muscle cell male adult (5...
...,...,...,...
7606,RNA:uterus,7606,RNA:uterus_7606
7607,RNA:uterus,7607,RNA:uterus_7607
7608,RNA:vagina,7608,RNA:vagina_7608
7609,RNA:vagina,7609,RNA:vagina_7609


In [9]:
import torch

ft = torch.load(f'/home/dnanexus/data_dir/{fz_file_prefix}3.pt')
ft.shape

/tmp/ipykernel_395766/491806379.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.


torch.Size([804299, 955])

In [10]:
ftd = pd.DataFrame(ft.numpy(), columns=descriptions['description_unique'])
ftd

description_unique,RNA:aortic smooth muscle cell male adult (21 years) and male adult (54 years)_6069,RNA:bladder microvascular endothelial cell male adult (46 years) and male adult (60 years)_6071,RNA:smooth muscle cell of bladder female adult (53 years) and male adult (62 years)_6073,RNA:bronchial epithelial cell female adult (40 years) and male adult (68 years)_6075,RNA:bronchial smooth muscle cell male adult (52 years) and male adult (59 years)_6077,RNA:endothelial cell of coronary artery female adult (41 years) and male adult (77 years)_6079,RNA:smooth muscle cell of the coronary artery female adult (53 years) and male adult (55 years)_6081,RNA:regular cardiac myocyte female adult (51 years) and male adult (48 years)_6083,RNA:dermis blood vessel endothelial cell female child (16 years) and male child (13 years)_6085,RNA:dermis lymphatic vessel endothelial cell female adult (45 years) and male child (6 years)_6087,...,RNA:testis_7601,RNA:testis_7602,RNA:thyroid_7603,RNA:thyroid_7604,RNA:thyroid_7605,RNA:uterus_7606,RNA:uterus_7607,RNA:vagina_7608,RNA:vagina_7609,RNA:vagina_7610
0,0.010889,0.008909,0.010505,-0.002259,0.008519,0.007509,0.008835,0.010555,0.008706,0.008822,...,-0.040870,-0.032999,-0.001240,-0.013278,-0.009414,-0.003362,-0.008902,-0.060381,-0.041510,-0.000225
1,-0.001857,-0.001407,-0.002555,-0.000448,-0.001432,-0.002151,-0.002150,-0.003684,-0.001362,-0.000504,...,0.053175,0.035775,0.011549,0.073606,0.073896,0.021276,0.077724,0.038422,0.038929,0.011645
2,0.008521,0.004973,0.009366,-0.004109,0.008820,0.005576,0.007647,0.009454,0.005610,0.005528,...,0.008485,0.003752,-0.006186,-0.033940,-0.043879,-0.001281,-0.003838,-0.054287,-0.038670,-0.002814
3,-0.030414,-0.023983,-0.034171,-0.014158,-0.029252,-0.024125,-0.029893,-0.031581,-0.023911,-0.025663,...,0.054911,0.044512,0.008223,0.024540,0.013839,0.014346,0.011843,0.031166,0.012209,0.006927
4,0.002517,0.004716,0.006598,0.004955,0.003229,0.004860,0.005084,0.007130,0.005162,0.004317,...,-0.054726,-0.032990,-0.010819,-0.061636,-0.068706,-0.020516,-0.087189,-0.037989,-0.032963,-0.011252
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
804294,-0.003309,-0.001551,-0.003040,-0.001931,-0.002565,-0.001746,-0.002213,-0.002124,-0.001514,-0.001254,...,-0.001159,-0.000552,-0.002865,-0.002168,-0.002137,-0.002485,-0.002233,-0.002906,-0.002947,-0.002303
804295,0.059224,0.065564,0.052025,0.052428,0.043916,0.057845,0.054275,0.055711,0.061668,0.062139,...,0.020116,0.016263,0.034829,0.025515,0.027641,0.027334,0.025758,0.028670,0.022112,0.028806
804296,-0.018815,-0.017196,-0.020113,-0.008149,-0.016664,-0.016981,-0.018458,-0.021956,-0.016405,-0.014861,...,-0.011144,-0.010533,-0.012648,-0.009740,-0.009578,-0.013452,-0.012774,-0.006796,-0.006863,-0.013055
804297,-0.003395,0.000054,-0.002874,0.000169,-0.003380,-0.000974,-0.002648,-0.002658,-0.000060,0.000202,...,-0.000283,-0.000008,0.000275,0.000325,0.000893,-0.000835,-0.000977,0.000435,0.000256,-0.000279


In [20]:
ftd['unique_id'] = a['unique_id'].to_numpy()
# ftd['id'] = ftd['unique_id'].str.split('_').str[0]
ftd['id'] = ftd['unique_id'].str.split('_').str[0:4].str.join(':')
ftd['region'] = ftd['unique_id'].str.split('_').str[4]
ftd_new = ftd.drop(columns=['unique_id'])
ftd_new

description_unique,RNA:aortic smooth muscle cell male adult (21 years) and male adult (54 years)_6069,RNA:bladder microvascular endothelial cell male adult (46 years) and male adult (60 years)_6071,RNA:smooth muscle cell of bladder female adult (53 years) and male adult (62 years)_6073,RNA:bronchial epithelial cell female adult (40 years) and male adult (68 years)_6075,RNA:bronchial smooth muscle cell male adult (52 years) and male adult (59 years)_6077,RNA:endothelial cell of coronary artery female adult (41 years) and male adult (77 years)_6079,RNA:smooth muscle cell of the coronary artery female adult (53 years) and male adult (55 years)_6081,RNA:regular cardiac myocyte female adult (51 years) and male adult (48 years)_6083,RNA:dermis blood vessel endothelial cell female child (16 years) and male child (13 years)_6085,RNA:dermis lymphatic vessel endothelial cell female adult (45 years) and male child (6 years)_6087,...,RNA:thyroid_7603,RNA:thyroid_7604,RNA:thyroid_7605,RNA:uterus_7606,RNA:uterus_7607,RNA:vagina_7608,RNA:vagina_7609,RNA:vagina_7610,id,region
0,0.010889,0.008909,0.010505,-0.002259,0.008519,0.007509,0.008835,0.010555,0.008706,0.008822,...,-0.001240,-0.013278,-0.009414,-0.003362,-0.008902,-0.060381,-0.041510,-0.000225,chr1:1019650:G:A,ENSG00000188157
1,-0.001857,-0.001407,-0.002555,-0.000448,-0.001432,-0.002151,-0.002150,-0.003684,-0.001362,-0.000504,...,0.011549,0.073606,0.073896,0.021276,0.077724,0.038422,0.038929,0.011645,chr1:1019763:A:T,ENSG00000188157
2,0.008521,0.004973,0.009366,-0.004109,0.008820,0.005576,0.007647,0.009454,0.005610,0.005528,...,-0.006186,-0.033940,-0.043879,-0.001281,-0.003838,-0.054287,-0.038670,-0.002814,chr1:1019639:G:A,ENSG00000188157
3,-0.030414,-0.023983,-0.034171,-0.014158,-0.029252,-0.024125,-0.029893,-0.031581,-0.023911,-0.025663,...,0.008223,0.024540,0.013839,0.014346,0.011843,0.031166,0.012209,0.006927,chr1:1019620:G:A,ENSG00000188157
4,0.002517,0.004716,0.006598,0.004955,0.003229,0.004860,0.005084,0.007130,0.005162,0.004317,...,-0.010819,-0.061636,-0.068706,-0.020516,-0.087189,-0.037989,-0.032963,-0.011252,chr1:1019690:C:G,ENSG00000188157
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
804294,-0.003309,-0.001551,-0.003040,-0.001931,-0.002565,-0.001746,-0.002213,-0.002124,-0.001514,-0.001254,...,-0.002865,-0.002168,-0.002137,-0.002485,-0.002233,-0.002906,-0.002947,-0.002303,chr22:50628450:G:C,ENSG00000100299
804295,0.059224,0.065564,0.052025,0.052428,0.043916,0.057845,0.054275,0.055711,0.061668,0.062139,...,0.034829,0.025515,0.027641,0.027334,0.025758,0.028670,0.022112,0.028806,chr22:50627787:G:C,ENSG00000100299
804296,-0.018815,-0.017196,-0.020113,-0.008149,-0.016664,-0.016981,-0.018458,-0.021956,-0.016405,-0.014861,...,-0.012648,-0.009740,-0.009578,-0.013452,-0.012774,-0.006796,-0.006863,-0.013055,chr22:50628287:T:C,ENSG00000100299
804297,-0.003395,0.000054,-0.002874,0.000169,-0.003380,-0.000974,-0.002648,-0.002658,-0.000060,0.000202,...,0.000275,0.000325,0.000893,-0.000835,-0.000977,0.000435,0.000256,-0.000279,chr22:50628542:G:A,ENSG00000100299


In [21]:
ftd_new.to_parquet(f'/home/dnanexus/data_dir/{fz_file_prefix}_ukbbgym.parquet')

In [22]:
!dx upload /home/dnanexus/data_dir/{fz_file_prefix}_ukbbgym.parquet --path project-Gyp4fvjJg0yFZ374KvP9bGFJ:/processed_data/ukbgym/flashzoi_scores/

[===========================================================>] Uploaded 3,095,526,466 of 3,095,526,466 bytes (100%) /home/dnanexus/data_dir/TSS_flashzoi_EURunrelated_variants_burden_test_olink_genes_GENCODEv40_ukbbgym.parquet=======>                                                  ] Uploaded 503,316,480 of 3,095,526,466 bytes (16%) /home/dnanexus/data_dir/TSS_flashzoi_EURunrelated_variants_burden_test_olink_genes_GENCODEv40_ukbbgym.parquet
ID                                file-J60BB70Jg0y52xjXz6Z6VKPv
Class                             file
Project                           project-Gyp4fvjJg0yFZ374KvP9bGFJ
Folder                            /processed_data/ukbgym/flashzoi_scores
Name                              TSS_flashzoi_EURunrelated_variants_burden_test_olink_genes_GENCODE
                                  v40_ukbbgym.parquet
State                             closing
Visibility                        visible
Types                             -
Properties                        -
T